# CNN1D — script retenu pour le mémoire

Architecture du chapitre 4.3.1 : 3 × Conv1D(256, noyau 2) + MaxPooling + dropout 0,25 → 0,5, Flatten, régularisation L2 (0,01), Adam (lr 1e-5), batch 64. Cellules de standardisation SG et SV, configurations WPDP et CPDP. La version imprimée dans le notebook contient la couche Dense(256) (329 730 paramètres) ; la variante simple s'obtient en la retirant.

**Statut :** Retenu (tableaux 9 et 10)

*Mémoire de maîtrise — Mireille Kafando, UQTR · 2024–2025*

> Notebook de recherche conçu pour Google Colab. Les chemins `/content/drive/MyDrive/bug/...` sont à adapter (voir `data/README.md`). Les sorties ont été retirées.


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import pandas as pd

# Liste des chemins de fichiers CSV pour l'entrainement
train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv', usecols=lambda x: x != 'name')

# Boucle pour lire les fichiers CSV
train_dfs = []
for file_path in train_file_paths:
    df = pd.read_csv(file_path, usecols=lambda x: x!= 'name')
    train_dfs.append(df)

# Concaténation des datasets en un seul DataFrame
df_train = pd.concat(train_dfs, ignore_index=True)

# Affichage des DataFrames concaténés
print(df_train)
print(df_val)


In [ ]:
import pandas as pd

# Chargement des DataFrames
df_paths = [
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv', usecols=lambda x: x != 'name')

dfs = [pd.read_csv(path, usecols=lambda x: x != 'name') for path in df_paths]

# Calcul de la moyenne et de l'écart-type pour df_val
means_val = df_val.mean(axis=0)
stds_val = df_val.std(axis=0)

# Supprimer les colonnes avec std > 1e-5 pour df_val
cols_to_drop_val = stds_val[abs(stds_val) > 1e-5].index.values
df_val = df_val.drop(cols_to_drop_val, axis=1)

# Normaliser df_val
cols_to_keep_val = df_val.columns
means_val = means_val[cols_to_keep_val]
stds_val = stds_val[cols_to_keep_val]
df_val[cols_to_keep_val] = (df_val[cols_to_keep_val] - means_val) / stds_val

remaining_columns_val = len(cols_to_keep_val)
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")

# Calcul de la moyenne et de l'écart-type pour chaque DataFrame dans dfs
means = []
stds = []
for i, df in enumerate(dfs):
    means.append(df.mean(axis=0))
    stds.append(df.std(axis=0))

# Standardisation des DataFrames et suppression des colonnes avec std > 1e-5
standardized_dfs = []
for i, df in enumerate(dfs):
    # Supprimer les colonnes avec std > 1e-5
    cols_to_drop = stds[i][abs(stds[i]) > 1e-5].index.values
    df = df.drop(cols_to_drop, axis=1)

    # Standardiser les colonnes restantes
    cols_to_keep = df.columns
    means[i] = means[i][cols_to_keep]
    stds[i] = stds[i][cols_to_keep]
    df[cols_to_keep] = (df[cols_to_keep] - means[i]) / stds[i]

    standardized_dfs.append(df)

    remaining_columns_training = len(cols_to_keep)
    print(f"Nombre de colonnes restantes dans le DataFrame df_v{i} : {remaining_columns_training}")

# Affichage des DataFrames standardisés
for i, df in enumerate(standardized_dfs):
    print(f'DataFrame df_v{i} standardisé:')
    print(df)
    print()

# Concaténation des DataFrames standardisés sauf df_val
df_train = pd.concat(standardized_dfs, ignore_index=True)

# Affichage du DataFrame concaténé
print(df_train)
print(df_val)


In [ ]:
import pandas as pd

# Chargement des DataFrames
df_paths = [
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv', usecols=lambda x: x != 'name')

dfs = [pd.read_csv(path, usecols=lambda x: x != 'name') for path in df_paths]

# Calcul de la moyenne et de l'écart-type pour chaque DataFrame
means = []
stds = []

means_val=df_val[df_val.columns[:-1]].mean(axis=0)
stds_val=df_val[df_val.columns[:-1]].std(axis=0)

# Calcul de la moyenne et de l'écart-type pour chaque DataFrame
for i, df in enumerate(dfs):
    df_without_last_col = df.iloc[:, :-1]
    means.append(df_without_last_col.mean(axis=0))
    stds.append(df_without_last_col.std(axis=0))

# Standardisation des DataFrames
standardized_dfs = []

for i, df in enumerate(dfs):
    df_without_last_col = df.iloc[:, :-1]
    df_standardized = (df_without_last_col - means[i]) / stds[i]

    # Ajout de la dernière colonne non modifiée
    df_standardized = pd.concat([df_standardized, df.iloc[:, -1]], axis=1)
    standardized_dfs.append(df_standardized)

df_val[df_val.columns[:-1]] = (df_val[df_val.columns[:-1]] - means_val) / stds_val

# Affichage des DataFrames standardisés
for i, df in enumerate(standardized_dfs):
    print(f'DataFrame df_v{i} standardisé:')
    print(df)
    print()

# Concaténation des DataFrames standardisés sauf df_val
df_train = pd.concat(standardized_dfs, ignore_index=True)

# Affichage du DataFrame concaténé
print(df_train)
print(df_val)

In [ ]:
# Fonction pour convertir les valeurs en binaire
def convert_to_binary(value):
    return 1 if value > 0 else 0

# Appliquer la fonction de conversion à la colonne 'bug'
df_train['bug'] = df_train['bug'].apply(convert_to_binary)
df_val['bug'] = df_val['bug'].apply(convert_to_binary)

# Afficher le dataset mis à jour
print(df_train)
print(df_val)
print(df_train.describe())


In [ ]:
from imblearn.over_sampling import SMOTE

# Use SMOTE to oversample the minority class in X_train
smote = SMOTE(sampling_strategy='auto')

# Resample X_train and y_train
X_train_resampled, y_train_resampled = smote.fit_resample(df_train.iloc[:, :-1].values, df_train.iloc[:, -1:].values)


x_train = pd.DataFrame(X_train_resampled, columns=df_train.columns[: -1])
y_train = pd.DataFrame(y_train_resampled, columns=df_train.columns[-1:])

df_training = pd.concat([x_train, y_train], axis=1)

print(df_training)




In [ ]:
import pandas as pd

# Calculer les moyennes et écarts types pour l'ensemble d'entraînement
means_tr = df_training[df_training.columns[:-1]].mean(axis=0)
stds_tr = df_training[df_training.columns[:-1]].std(axis=0)

# Calculer les moyennes et écarts types pour l'ensemble de validation
means_val = df_val[df_val.columns[:-1]].mean(axis=0)
stds_val = df_val[df_val.columns[:-1]].std(axis=0)

# Supprimer les colonnes avec std > 1e-5
cols_to_drop_training = stds_tr[abs(stds_tr) <= 1e-5].index.values
cols_to_drop_val = stds_val[abs(stds_val) <= 1e-5].index.values

df_training = df_training.drop(cols_to_drop_training, axis=1)
df_val = df_val.drop(cols_to_drop_val, axis=1)

# Sélectionner les colonnes restantes
cols_to_keep_training = df_training.columns[:-1]
cols_to_keep_val = df_val.columns[:-1]

remaining_columns_training = len(cols_to_keep_training)
remaining_columns_val = len(cols_to_keep_val)

# Normaliser les colonnes restantes
df_training[cols_to_keep_training] = (df_training[cols_to_keep_training] - means_tr[cols_to_keep_training]) / stds_tr[cols_to_keep_training]
df_val[cols_to_keep_val] = (df_val[cols_to_keep_val] - means_val[cols_to_keep_val]) / stds_val[cols_to_keep_val]

print(df_training)
print(df_val)

print(f"Nombre de colonnes restantes dans l'ensemble d'entraînement : {remaining_columns_training}")
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")


In [ ]:
import pandas as pd

# Calculer les moyennes et écarts types pour l'ensemble d'entraînement
means_tr = df_training[df_training.columns[:-1]].mean(axis=0)
stds_tr = df_training[df_training.columns[:-1]].std(axis=0)

# Calculer les moyennes et écarts types pour l'ensemble de validation
means_val = df_val[df_val.columns[:-1]].mean(axis=0)
stds_val = df_val[df_val.columns[:-1]].std(axis=0)

# Supprimer les colonnes avec std > 1e-5
cols_to_drop_training = stds_tr[abs(stds_tr) <= 1e-5].index.values
cols_to_drop_val = stds_val[abs(stds_val) <= 1e-5].index.values

df_training_filtered = df_training.drop(cols_to_drop_training, axis=1)
df_val_filtered = df_val.drop(cols_to_drop_val, axis=1)

# Sélectionner les colonnes restantes
cols_to_keep_training = df_training_filtered.columns[:-1]
cols_to_keep_val = df_val_filtered.columns[:-1]

remaining_columns_training = len(cols_to_keep_training)
remaining_columns_val = len(cols_to_keep_val)

# Normaliser les colonnes restantes
df_training_filtered[cols_to_keep_training] = (df_training_filtered[cols_to_keep_training] - means_tr[cols_to_keep_training]) / stds_tr[cols_to_keep_training]
df_val_filtered[cols_to_keep_val] = (df_val_filtered[cols_to_keep_val] - means_val[cols_to_keep_val]) / stds_val[cols_to_keep_val]

print(df_training_filtered)
print(df_val_filtered)

print(f"Nombre de colonnes restantes dans l'ensemble d'entraînement : {remaining_columns_training}")
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")


In [ ]:
import pandas as pd

# Calculer les moyennes et écarts types pour l'ensemble d'entraînement
means_tr = df_training[df_training.columns[:-1]].mean(axis=0)
stds_tr = df_training[df_training.columns[:-1]].std(axis=0)

# Calculer les moyennes et écarts types pour l'ensemble de validation
means_val = df_val[df_val.columns[:-1]].mean(axis=0)
stds_val = df_val[df_val.columns[:-1]].std(axis=0)

# Supprimer les colonnes avec std > 1e-5
cols_to_drop_training = stds_tr[abs(stds_tr) > 1e-5].index.values
cols_to_drop_val = stds_val[abs(stds_val) > 1e-5].index.values

# Mise à jour des DataFrames en supprimant les colonnes sélectionnées
df_training = df_training.drop(cols_to_drop_training, axis=1)
df_val = df_val.drop(cols_to_drop_val, axis=1)

# Sélectionner les colonnes restantes
cols_to_keep_training = df_training.columns[:-1]
cols_to_keep_val = df_val.columns[:-1]

remaining_columns_training = len(cols_to_keep_training)
remaining_columns_val = len(cols_to_keep_val)

# Normaliser les colonnes restantes
df_training[cols_to_keep_training] = (df_training[cols_to_keep_training] - means_tr[cols_to_keep_training]) / stds_tr[cols_to_keep_training]
df_val[cols_to_keep_val] = (df_val[cols_to_keep_val] - means_val[cols_to_keep_val]) / stds_val[cols_to_keep_val]

print(df_training)
print(df_val)

print(f"Nombre de colonnes restantes dans l'ensemble d'entraînement : {remaining_columns_training}")
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")


In [ ]:

means_tr =df_training[df_training.columns[:-1]].mean(axis=0)

stds_tr = df_training[df_training.columns[:-1]].std(axis=0)

means_val=df_val[df_val.columns[:-1]].mean(axis=0)

stds_val=df_val[df_val.columns[:-1]].std(axis=0)

df_training[df_training.columns[:-1]] = (df_training[df_training.columns[:-1]] - means_tr) / stds_tr
df_val[df_val.columns[:-1]] = (df_val[df_val.columns[:-1]] - means_val) / stds_val


print(df_training)
print(df_val)


In [ ]:
means_tr =df_training[df_training.columns[:-1]].mean(axis=0)

stds_tr = df_training[df_training.columns[:-1]].std(axis=0)

means_val=df_val[df_val.columns[:-1]].mean(axis=0)

stds_val=df_val[df_val.columns[:-1]].std(axis=0)


# Use the stds_tr variable directly, not as an attribute
df_training = df_training.drop(stds_tr[abs(stds_tr) <= 1e-5].index.values, axis=1)
df_val = df_val.drop(stds_val[abs(stds_val) <= 1e-5].index.values, axis=1)
# Exclude the 'bug' column when dropping columns

df_training = df_training.drop(stds_tr[abs(stds_tr) <= 1e-5].index.values, axis=1)
df_val = df_val.drop(stds_val[abs(stds_val) <= 1e-5].index.values, axis=1)

# Exclude the 'bug' column when selecting columns to keep
cols_to_keep_training = df_training.columns[:-1][stds_tr <= 1e-5]
cols_to_keep_val = df_val.columns[:-1][stds_val <= 1e-5]

remaining_columns_training = len(cols_to_keep_training)
remaining_columns_val = len(cols_to_keep_val)

df_training[df_training.columns[:-1]] = (df_training[df_training.columns[:-1]] - means_tr) / stds_tr
df_val[df_val.columns[:-1]] = (df_val[df_val.columns[:-1]] - means_val) / stds_val


print(df_training)
print(df_val)

In [ ]:

train = pd.get_dummies(df_training, dtype= int, columns=['bug'], prefix=['bug_'])
val = pd.get_dummies(df_val, dtype= int, columns=['bug'], prefix=['bug_'])

print(train.describe())

x_training = train.iloc[:, :-2]
y_training = train.iloc[:, -2:]

#print(x_training)

x_val = val.iloc[:, :-2]
y_val = val.iloc[:, -2:]

print(val.describe())



In [ ]:

x_training = x_training.values
y_training = y_training.values
x_val =  x_val.values
y_val =  y_val.values


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(256, 2, activation='relu', input_shape=(20, 1)),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Conv1D(256, 2, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Conv1D(256, 2, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01))
    tf.keras.layers.Dense(2, activation='softmax', kernel_regularizer=regularizers.l2(0.01))
])
model.summary()


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Compiler le modèle
model.compile(optimizer=Adam(learning_rate=0.00001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Configuration de l'arrêt précoce
early_stopping = EarlyStopping(monitor='val_loss', patience=150, restore_best_weights=True)

# Entraîner le modèle avec l'arrêt précoce
history = model.fit(x_training, y_training, epochs=500, batch_size=64, validation_data=(x_val, y_val), callbacks=[early_stopping])


In [ ]:
import matplotlib.pyplot as plt

# Graphique de l'accuracy
plt.figure(figsize=(4, 2))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Graphique de la loss
plt.figure(figsize=(4, 2))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


y_pred = model.predict(x_val)

# Convertir les prédictions en classes
y_pred_classes = (y_pred > 0.5).astype(int)

# Convertir les étiquettes en classes
y_test_classes = (y_val > 0.5).astype(int)

# Calculer la matrice de confusion
cm = confusion_matrix(y_test_classes.argmax(axis=1), y_pred_classes.argmax(axis=1))

print(cm)

# Calculer la sensibilité (recall) et la spécificité
TN, FP, FN, TP = cm.ravel()

sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)

# Calculer le G-mean
g_mean = np.sqrt(sensitivity * specificity)
print(f'G-mean: {g_mean}')

# Visualiser la matrice de confusion
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Pas Bug', 'Bug'], yticklabels=['Pas Bug', 'Bug'])
plt.xlabel('Prédictions')
plt.ylabel('Vraies Classes')
plt.title('Matrice de Confusion')
plt.show()

In [ ]:
from sklearn.metrics import classification_report

# Faire des prédictions
predictions = model.predict(x_val)

# Supposons que notre modèle prédit des probabilités et que nous utilisons un seuil de 0.5
predictions_binary = (predictions > 0.5).astype(int)

# Générer le rapport de classification
report = classification_report(y_val, predictions_binary)

# Afficher le rapport de classification
print("Classification Report:")
print(report)

In [ ]:
from sklearn.model_selection import KFold
import numpy as np


losses = []
accuracies = []

kfold = KFold(n_splits=15, shuffle=True, random_state=42)

for train_index, val_index in kfold.split(x_training):
    x_train_fold, X_val_fold = x_training[train_index], x_training[val_index]
    y_train_fold, y_val_fold = y_training[train_index], y_training[val_index]

    model.fit(x_train_fold, y_train_fold, epochs=50, batch_size=64, verbose=0)
    loss, accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    losses.append(loss)
    accuracies.append(accuracy)
    print(f"Loss: {loss}, Accuracy: {accuracy}")

# Calculer la moyenne et l'écart-type
mean_loss = np.mean(losses)
std_loss = np.std(losses)
mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

print(f"Mean Loss: {mean_loss}, Standard Deviation of Loss: {std_loss}")
print(f"Mean Accuracy: {mean_accuracy}, Standard Deviation of Accuracy: {std_accuracy}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import pandas as pd

# Chargement des DataFrames
df_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.7.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-3.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.7.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.3.csv',


]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.4.4.csv', usecols=lambda x: x != 'name')

dfs = [pd.read_csv(path, usecols=lambda x: x != 'name') for path in df_paths]

# Calcul de la moyenne et de l'écart-type pour df_val
means_val = df_val[df_val.columns[:-1]].mean(axis=0)
stds_val = df_val[df_val.columns[:-1]].std(axis=0)

# Supprimer les colonnes avec std > 1e-5 pour df_val
cols_to_drop_val = stds_val[abs(stds_val) <= 1e-5].index.values
df_val = df_val.drop(cols_to_drop_val, axis=1)

# Normaliser df_val
cols_to_keep_val = df_val.columns[:-1]
means_val = means_val[cols_to_keep_val]
stds_val = stds_val[cols_to_keep_val]
df_val[cols_to_keep_val] = (df_val[cols_to_keep_val] - means_val) / stds_val

remaining_columns_val = len(cols_to_keep_val)
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")

# Calcul de la moyenne et de l'écart-type pour chaque DataFrame dans dfs
means = []
stds = []
for i, df in enumerate(dfs):
    df_without_last_col = df.iloc[:, :-1]
    means.append(df_without_last_col.mean(axis=0))
    stds.append(df_without_last_col.std(axis=0))

# Standardisation des DataFrames et suppression des colonnes avec std > 1e-5
standardized_dfs = []
for i, df in enumerate(dfs):
    df_without_last_col = df.iloc[:, :-1]

    # Supprimer les colonnes avec std > 1e-5
    cols_to_drop = stds[i][abs(stds[i]) <= 1e-5].index.values
    df = df.drop(cols_to_drop, axis=1)

    # Normaliser les colonnes restantes
    cols_to_keep = df.columns[:-1]
    means[i] = means[i][cols_to_keep]
    stds[i] = stds[i][cols_to_keep]
    df[cols_to_keep] = (df[cols_to_keep] - means[i]) / stds[i]

    # Ajout de la dernière colonne non modifiée
    last_col = df.iloc[:, -1]
    df_standardized = pd.concat([df[cols_to_keep], last_col], axis=1)
    standardized_dfs.append(df_standardized)

    remaining_columns_training = len(cols_to_keep)
    print(f"Nombre de colonnes restantes dans le DataFrame df_v{i} : {remaining_columns_training}")

# Affichage des DataFrames standardisés
for i, df in enumerate(standardized_dfs):
    print(f'DataFrame df_v{i} standardisé:')
    print(df)
    print()

# Concaténation des DataFrames standardisés sauf df_val
df_train = pd.concat(standardized_dfs, ignore_index=True)

# Affichage du DataFrame concaténé
print(df_train)
print(df_val)


In [ ]:
import pandas as pd

# Liste des chemins de fichiers CSV pour l'entrainement
train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv', usecols=lambda x: x != 'name')

# Boucle pour lire les fichiers CSV
train_dfs = []
for file_path in train_file_paths:
    df = pd.read_csv(file_path, usecols=lambda x: x!= 'name')
    train_dfs.append(df)

# Concaténation des datasets en un seul DataFrame
df_train = pd.concat(train_dfs, ignore_index=True)

# Affichage des DataFrames concaténés
print(df_train)
print(df_val)


In [ ]:
# Fonction pour convertir les valeurs en binaire
def convert_to_binary(value):
    return 1 if value > 0 else 0

# Appliquer la fonction de conversion à la colonne 'bug'
df_train['bug'] = df_train['bug'].apply(convert_to_binary)
df_val['bug'] = df_val['bug'].apply(convert_to_binary)

# Afficher le dataset mis à jour
print(df_train)
print(df_val)
print(df_train.describe())


In [ ]:
!pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

# Use SMOTE to oversample the minority class in X_train
smote = SMOTE(sampling_strategy='auto')

# Resample X_train and y_train
X_train_resampled, y_train_resampled = smote.fit_resample(df_train.iloc[:, :-1].values, df_train.iloc[:, -1:].values)


x_train = pd.DataFrame(X_train_resampled, columns=df_train.columns[: -1])
y_train = pd.DataFrame(y_train_resampled, columns=df_train.columns[-1:])

df_training = pd.concat([x_train, y_train], axis=1)

print(df_training)


In [ ]:
print(df_train.describe())

In [ ]:
import pandas as pd

# Calculer les moyennes et écarts types pour l'ensemble d'entraînement
means_tr = df_training[df_training.columns[:-1]].mean(axis=0)
stds_tr = df_training[df_training.columns[:-1]].std(axis=0)

# Calculer les moyennes et écarts types pour l'ensemble de validation
means_val = df_val[df_val.columns[:-1]].mean(axis=0)
stds_val = df_val[df_val.columns[:-1]].std(axis=0)

# Supprimer les colonnes avec std > 1e-5
cols_to_drop_training = stds_tr[abs(stds_tr) <= 1e-4].index.values
cols_to_drop_val = stds_val[abs(stds_val) <= 1e-4].index.values

df_training = df_training.drop(cols_to_drop_training, axis=1)
df_val = df_val.drop(cols_to_drop_val, axis=1)

# Sélectionner les colonnes restantes
cols_to_keep_training = df_training.columns[:-1]
cols_to_keep_val = df_val.columns[:-1]

remaining_columns_training = len(cols_to_keep_training)
remaining_columns_val = len(cols_to_keep_val)

# Normaliser les colonnes restantes
df_training[cols_to_keep_training] = (df_training[cols_to_keep_training] - means_tr[cols_to_keep_training]) / stds_tr[cols_to_keep_training]
df_val[cols_to_keep_val] = (df_val[cols_to_keep_val] - means_val[cols_to_keep_val]) / stds_val[cols_to_keep_val]

print(df_training)
print(df_val)

print(f"Nombre de colonnes restantes dans l'ensemble d'entraînement : {remaining_columns_training}")
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")


In [ ]:

train = pd.get_dummies(df_training, dtype= int, columns=['bug'], prefix=['bug_'])
val = pd.get_dummies(df_val, dtype= int, columns=['bug'], prefix=['bug_'])

print(train.describe())

x_training = train.iloc[:, :-2]
y_training = train.iloc[:, -2:]

#print(x_training)

x_val = val.iloc[:, :-2]
y_val = val.iloc[:, -2:]

print(val.describe())

In [ ]:
x_training = x_training.values
y_training = y_training.values
x_val =  x_val.values
y_val =  y_val.values


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(16, 2, activation='relu', input_shape=(20, 1)),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.22191711633138964),
    tf.keras.layers.Conv1D(32, 2, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.22191711633138964),
    tf.keras.layers.Conv1D(64, 2, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.22191711633138964),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.00512825663132412)),  # Ajout de la régularisation L2
    tf.keras.layers.Dense(2, activation='softmax', kernel_regularizer=regularizers.l2(0.00512825663132412))  # Ajout de la régularisation L2
])
model.summary()


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Compiler le modèle
model.compile(optimizer=Adam(learning_rate=0.000464950189674214),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Configuration de l'arrêt précoce
early_stopping = EarlyStopping(monitor='val_loss', patience=450, restore_best_weights=True)

# Entraîner le modèle avec l'arrêt précoce
history = model.fit(x_training, y_training, epochs=500, batch_size=8, validation_data=(x_val, y_val), callbacks=[early_stopping])


In [ ]:
import matplotlib.pyplot as plt

# Graphique de l'accuracy
plt.figure(figsize=(4, 2))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Graphique de la loss
plt.figure(figsize=(4, 2))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


y_pred = model.predict(x_val)

# Convertir les prédictions en classes
y_pred_classes = (y_pred > 0.5).astype(int)

# Convertir les étiquettes en classes
y_test_classes = (y_val > 0.5).astype(int)

# Calculer la matrice de confusion
cm = confusion_matrix(y_test_classes.argmax(axis=1), y_pred_classes.argmax(axis=1))

print(cm)

# Calculer la sensibilité (recall) et la spécificité
TN, FP, FN, TP = cm.ravel()

sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)

# Calculer le G-mean
g_mean = np.sqrt(sensitivity * specificity)
print(f'G-mean: {g_mean}')

# Visualiser la matrice de confusion
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Pas Bug', 'Bug'], yticklabels=['Pas Bug', 'Bug'])
plt.xlabel('Prédictions')
plt.ylabel('Vraies Classes')
plt.title('Matrice de Confusion')
plt.show()

In [ ]:
from sklearn.metrics import classification_report

# Faire des prédictions
predictions = model.predict(x_val)

# Supposons que notre modèle prédit des probabilités et que nous utilisons un seuil de 0.5
predictions_binary = (predictions > 0.5).astype(int)

# Générer le rapport de classification
report = classification_report(y_val, predictions_binary)

# Afficher le rapport de classification
print("Classification Report:")
print(report)

In [ ]:
from sklearn.model_selection import KFold
import numpy as np


losses = []
accuracies = []

kfold = KFold(n_splits=15, shuffle=True, random_state=42)

for train_index, val_index in kfold.split(x_training):
    x_train_fold, X_val_fold = x_training[train_index], x_training[val_index]
    y_train_fold, y_val_fold = y_training[train_index], y_training[val_index]

    model.fit(x_train_fold, y_train_fold, epochs=50, batch_size=64, verbose=0)
    loss, accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    losses.append(loss)
    accuracies.append(accuracy)
    print(f"Loss: {loss}, Accuracy: {accuracy}")

# Calculer la moyenne et l'écart-type
mean_loss = np.mean(losses)
std_loss = np.std(losses)
mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

print(f"Mean Loss: {mean_loss}, Standard Deviation of Loss: {std_loss}")
print(f"Mean Accuracy: {mean_accuracy}, Standard Deviation of Accuracy: {std_accuracy}")

In [ ]:
import pandas as pd

# Liste des chemins de fichiers CSV pour l'entrainement
train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv', usecols=lambda x: x != 'name')

# Boucle pour lire les fichiers CSV
train_dfs = []
for file_path in train_file_paths:
    df = pd.read_csv(file_path, usecols=lambda x: x!= 'name')
    train_dfs.append(df)

# Concaténation des datasets en un seul DataFrame
df_train = pd.concat(train_dfs, ignore_index=True)

# Affichage des DataFrames concaténés
print(df_train)
print(df_val)


In [ ]:
import pandas as pd

# Liste des chemins de fichiers CSV pour l'entrainement
train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv', usecols=lambda x: x != 'name')

# Boucle pour lire les fichiers CSV
train_dfs = []
for file_path in train_file_paths:
    df = pd.read_csv(file_path, usecols=lambda x: x!= 'name')
    train_dfs.append(df)

# Concaténation des datasets en un seul DataFrame
df_train = pd.concat(train_dfs, ignore_index=True)

# Affichage des DataFrames concaténés
print(df_train)
print(df_val)


In [ ]:
import pandas as pd

# Liste des chemins de fichiers CSV pour l'entrainement
train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv', usecols=lambda x: x != 'name')

# Boucle pour lire les fichiers CSV
train_dfs = []
for file_path in train_file_paths:
    df = pd.read_csv(file_path, usecols=lambda x: x!= 'name')
    train_dfs.append(df)

# Concaténation des datasets en un seul DataFrame
df_train = pd.concat(train_dfs, ignore_index=True)

# Affichage des DataFrames concaténés
print(df_train)
print(df_val)

# Chargement des DataFrames
df_paths = [
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
]

df_val = pd.read_csv('/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv', usecols=lambda x: x != 'name')

dfs = [pd.read_csv(path, usecols=lambda x: x != 'name') for path in df_paths]

# Calcul de la moyenne et de l'écart-type pour df_val
means_val = df_val.mean(axis=0)
stds_val = df_val.std(axis=0)

# Supprimer les colonnes avec std > 1e-5 pour df_val
cols_to_drop_val = stds_val[abs(stds_val) > 1e-5].index.values
df_val = df_val.drop(cols_to_drop_val, axis=1)

# Standardiser df_val
cols_to_keep_val = df_val.columns
means_val = means_val[cols_to_keep_val]
stds_val = stds_val[cols_to_keep_val]
df_val[cols_to_keep_val] = (df_val[cols_to_keep_val] - means_val) / stds_val

remaining_columns_val = len(cols_to_keep_val)
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")

# Calcul de la moyenne et de l'écart-type pour chaque DataFrame dans dfs
means = []
stds = []
for i, df in enumerate(dfs):
    means.append(df.mean(axis=0))
    stds.append(df.std(axis=0))

# Standardisation S2 des DataFrames et suppression des colonnes avec std > 1e-5
standardized_dfs = []
for i, df in enumerate(dfs):
    # Supprimer les colonnes avec std > 1e-5
    cols_to_drop = stds[i][abs(stds[i]) > 1e-5].index.values
    df = df.drop(cols_to_drop, axis=1)

    # Standardiser les colonnes restantes
    cols_to_keep = df.columns
    means[i] = means[i][cols_to_keep]
    stds[i] = stds[i][cols_to_keep]
    df[cols_to_keep] = (df[cols_to_keep] - means[i]) / stds[i]

    standardized_dfs.append(df)

    remaining_columns_training = len(cols_to_keep)
    print(f"Nombre de colonnes restantes dans le DataFrame df_v{i} : {remaining_columns_training}")

# Affichage des DataFrames standardisés
for i, df in enumerate(standardized_dfs):
    print(f'DataFrame df_v{i} standardisé:')
    print(df)
    print()

# Concaténation des DataFrames standardisés sauf df_val
df_train = pd.concat(standardized_dfs, ignore_index=True)

# Affichage du DataFrame concaténé
print(df_train)
print(df_val)

# Fonction pour convertir les valeurs en binaire
def convert_to_binary(value):
    return 1 if value > 0 else 0

# Appliquer la fonction de conversion à la colonne 'bug'
df_train['bug'] = df_train['bug'].apply(convert_to_binary)
df_val['bug'] = df_val['bug'].apply(convert_to_binary)

# Afficher le dataset mis à jour
print(df_train)
print(df_val)
print(df_train.describe())


# Calculer les moyennes et écarts types pour l'ensemble d'entraînement
means_tr = df_training[df_training.columns[:-1]].mean(axis=0)
stds_tr = df_training[df_training.columns[:-1]].std(axis=0)

# Calculer les moyennes et écarts types pour l'ensemble de validation
means_val = df_val[df_val.columns[:-1]].mean(axis=0)
stds_val = df_val[df_val.columns[:-1]].std(axis=0)

# Supprimer les colonnes avec std > 1e-5
cols_to_drop_training = stds_tr[abs(stds_tr) > 1e-5].index.values
cols_to_drop_val = stds_val[abs(stds_val) > 1e-5].index.values

# Mise à jour des DataFrames en supprimant les colonnes sélectionnées
df_training = df_training.drop(cols_to_drop_training, axis=1)
df_val = df_val.drop(cols_to_drop_val, axis=1)

# Sélectionner les colonnes restantes
cols_to_keep_training = df_training.columns[:-1]
cols_to_keep_val = df_val.columns[:-1]

remaining_columns_training = len(cols_to_keep_training)
remaining_columns_val = len(cols_to_keep_val)

# standardisation les colonnes restantes
df_training[cols_to_keep_training] = (df_training[cols_to_keep_training] - means_tr[cols_to_keep_training]) / stds_tr[cols_to_keep_training]
df_val[cols_to_keep_val] = (df_val[cols_to_keep_val] - means_val[cols_to_keep_val]) / stds_val[cols_to_keep_val]

print(df_training)
print(df_val)

print(f"Nombre de colonnes restantes dans l'ensemble d'entraînement : {remaining_columns_training}")
print(f"Nombre de colonnes restantes dans l'ensemble de validation : {remaining_columns_val}")

means_tr =df_training[df_training.columns[:-1]].mean(axis=0)

stds_tr = df_training[df_training.columns[:-1]].std(axis=0)

means_val=df_val[df_val.columns[:-1]].mean(axis=0)

stds_val=df_val[df_val.columns[:-1]].std(axis=0)

df_training[df_training.columns[:-1]] = (df_training[df_training.columns[:-1]] - means_tr) / stds_tr
df_val[df_val.columns[:-1]] = (df_val[df_val.columns[:-1]] - means_val) / stds_val


print(df_training)
print(df_val)

train = pd.get_dummies(df_training, dtype= int, columns=['bug'], prefix=['bug_'])
val = pd.get_dummies(df_val, dtype= int, columns=['bug'], prefix=['bug_'])

print(train.describe())

x_training = train.iloc[:, :-2]
y_training = train.iloc[:, -2:]

#print(x_training)

x_val = val.iloc[:, :-2]
y_val = val.iloc[:, -2:]

print(val.describe())

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(256, 2, activation='relu', input_shape=(20, 1)),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Conv1D(256, 2, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Conv1D(256, 2, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01)), # une seule couche pour variante simple
    #tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01)) deuxième couche pour variante dense
    tf.keras.layers.Dense(2, activation='softmax', kernel_regularizer=regularizers.l2(0.01))
])
model.summary()

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Compiler le modèle
model.compile(optimizer=Adam(learning_rate=0.000464950189674214),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Configuration de l'arrêt précoce
early_stopping = EarlyStopping(monitor='val_loss', patience=450, restore_best_weights=True)

# Entraîner le modèle avec l'arrêt précoce
history = model.fit(x_training, y_training, epochs=500, batch_size=8, validation_data=(x_val, y_val), callbacks=[early_stopping])

import matplotlib.pyplot as plt

# Graphique de l'accuracy
plt.figure(figsize=(4, 2))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Graphique de la loss
plt.figure(figsize=(4, 2))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


y_pred = model.predict(x_val)

# Convertir les prédictions en classes
y_pred_classes = (y_pred > 0.5).astype(int)

# Convertir les étiquettes en classes
y_test_classes = (y_val > 0.5).astype(int)

# Calculer la matrice de confusion
cm = confusion_matrix(y_test_classes.argmax(axis=1), y_pred_classes.argmax(axis=1))

print(cm)

# Calculer la sensibilité (recall) et la spécificité
TN, FP, FN, TP = cm.ravel()

sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)

# Calculer le G-mean
g_mean = np.sqrt(sensitivity * specificity)
print(f'G-mean: {g_mean}')

# Visualiser la matrice de confusion
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Pas Bug', 'Bug'], yticklabels=['Pas Bug', 'Bug'])
plt.xlabel('Prédictions')
plt.ylabel('Vraies Classes')
plt.title('Matrice de Confusion')
plt.show()

from sklearn.metrics import classification_report

# Faire des prédictions
predictions = model.predict(x_val)

# Supposons que notre modèle prédit des probabilités et que nous utilisons un seuil de 0.5
predictions_binary = (predictions > 0.5).astype(int)

# Générer le rapport de classification
report = classification_report(y_val, predictions_binary)

# Afficher le rapport de classification
print("Classification Report:")
print(report)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch

def create_cnn1d_architecture_diagram():
    """CNN1D Architecture Diagram with Standardization Strategies"""

    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.rcParams['font.size'] = 8

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 10)
    ax.axis('off')

    # Title
    ax.text(6, 9.5, 'CNN1D ARCHITECTURE FOR SOFTWARE DEFECT PREDICTION',
            ha='center', va='center', fontweight='bold', fontsize=11)

    # ==================== STANDARDIZATION STRATEGIES ====================
    ax.text(6, 8.8, 'STANDARDIZATION STRATEGIES', ha='center', va='center',
            fontweight='bold', fontsize=10, color='navy')

    # S1 Strategy
    s1_box = FancyBboxPatch((1, 7.5), 3, 0.8,
                           boxstyle="round,pad=0.02",
                           facecolor='#E3F2FD',
                           edgecolor='#1976D2', linewidth=1.5)
    ax.add_patch(s1_box)
    ax.text(2.5, 7.9, 'STRATEGY S1', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(2.5, 7.6, 'Global standardization',
            ha='center', va='center', fontsize=8)

    # S2 Strategy
    s2_box = FancyBboxPatch((5, 7.5), 3, 0.8,
                           boxstyle="round,pad=0.02",
                           facecolor='#E8F5E8',
                           edgecolor='#388E3C', linewidth=1.5)
    ax.add_patch(s2_box)
    ax.text(6.5, 7.9, 'STRATEGY S2', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(6.5, 7.6, 'Per-project standardization',
            ha='center', va='center', fontsize=8)

    # S3 Strategy
    s3_box = FancyBboxPatch((9, 7.5), 3, 0.8,
                           boxstyle="round,pad=0.02",
                           facecolor='#FFF3CD',
                           edgecolor='#FFA000', linewidth=1.5)
    ax.add_patch(s3_box)
    ax.text(10.5, 7.9, 'STRATEGY S3-ROBUST', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(10.5, 7.6, 'Robust preprocessing',
            ha='center', va='center', fontsize=8)

    # ==================== INPUT LAYER ====================
    input_box = FancyBboxPatch((1, 6.2), 2, 0.7,
                             boxstyle="round,pad=0.02",
                             facecolor='#F5F5F5',
                             edgecolor='gray', linewidth=1)
    ax.add_patch(input_box)
    ax.text(2, 6.55, 'INPUT', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(2, 6.25, '20 CK metrics',
            ha='center', va='center', fontsize=8)

    # ==================== CONVOLUTIONAL BLOCKS ====================
    ax.text(6, 6.8, 'CONVOLUTIONAL BLOCKS (3x)', ha='center', va='center',
            fontweight='bold', fontsize=10, color='purple')

    # Conv Block 1
    conv1_box = FancyBboxPatch((3.5, 5.5), 2.5, 0.8,
                              boxstyle="round,pad=0.02",
                              facecolor='#E1BEE7',
                              edgecolor='#7B1FA2', linewidth=1.5)
    ax.add_patch(conv1_box)
    ax.text(4.75, 5.85, 'CONV1D BLOCK 1', ha='center', va='center',
            fontweight='bold', fontsize=8)
    ax.text(4.75, 5.55, '256 filters • kernel=2\nDropout: 0.25',
            ha='center', va='center', fontsize=7)

    # Conv Block 2
    conv2_box = FancyBboxPatch((3.5, 4.5), 2.5, 0.8,
                              boxstyle="round,pad=0.02",
                              facecolor='#C8E6C9',
                              edgecolor='#388E3C', linewidth=1.5)
    ax.add_patch(conv2_box)
    ax.text(4.75, 4.85, 'CONV1D BLOCK 2', ha='center', va='center',
            fontweight='bold', fontsize=8)
    ax.text(4.75, 4.55, '256 filters • kernel=2\nDropout: 0.5',
            ha='center', va='center', fontsize=7)

    # Conv Block 3
    conv3_box = FancyBboxPatch((3.5, 3.5), 2.5, 0.8,
                              boxstyle="round,pad=0.02",
                              facecolor='#BBDEFB',
                              edgecolor='#1976D2', linewidth=1.5)
    ax.add_patch(conv3_box)
    ax.text(4.75, 3.85, 'CONV1D BLOCK 3', ha='center', va='center',
            fontweight='bold', fontsize=8)
    ax.text(4.75, 3.55, '256 filters • kernel=2\nDropout: 0.5',
            ha='center', va='center', fontsize=7)

    # MaxPooling annotation
    ax.text(6.5, 5, 'MAXPOOLING\n(size=2)\n↕ after each block',
            ha='center', va='center', fontsize=7,
            bbox=dict(boxstyle="round,pad=0.3", facecolor='#FFF3CD'))

    # ==================== FLATTEN LAYER ====================
    flatten_box = FancyBboxPatch((3.5, 2.5), 2.5, 0.7,
                               boxstyle="round,pad=0.02",
                               facecolor='#FFE0B2',
                               edgecolor='#F57C00', linewidth=1.5)
    ax.add_patch(flatten_box)
    ax.text(4.75, 2.85, 'FLATTEN', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(4.75, 2.55, 'Feature maps → Vector',
            ha='center', va='center', fontsize=7)

    # ==================== DENSE LAYER VARIANTS ====================
    ax.text(8.5, 6.8, 'ARCHITECTURE VARIANTS', ha='center', va='center',
            fontweight='bold', fontsize=10, color='#D32F2F')

    # CNN1D-Simple Variant
    simple_box = FancyBboxPatch((7, 5.5), 3, 1.2,
                              boxstyle="round,pad=0.02",
                              facecolor='#E8F5E8',
                              edgecolor='#388E3C', linewidth=2)
    ax.add_patch(simple_box)
    ax.text(8.5, 6.0, 'CNN1D-SIMPLE', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(8.5, 5.7, 'Single Dense Layer\n256 neurons',
            ha='center', va='center', fontsize=8)
    ax.text(8.5, 5.4, '~180K parameters\nL2 regularization',
            ha='center', va='center', fontsize=7)

    # CNN1D-Dense Variant
    dense_box = FancyBboxPatch((7, 3.5), 3, 1.2,
                             boxstyle="round,pad=0.02",
                             facecolor='#FFEBEE',
                             edgecolor='#D32F2F', linewidth=2)
    ax.add_patch(dense_box)
    ax.text(8.5, 4.0, 'CNN1D-DENSE', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(8.5, 3.7, 'Two Dense Layers\n256 neurons each',
            ha='center', va='center', fontsize=8)
    ax.text(8.5, 3.4, '~240K parameters\nL2 regularization',
            ha='center', va='center', fontsize=7)

    # ==================== OUTPUT LAYER ====================
    output_box = FancyBboxPatch((3.5, 1), 2.5, 0.7,
                              boxstyle="round,pad=0.02",
                              facecolor='#4CAF50',
                              edgecolor='#2E7D32', linewidth=2)
    ax.add_patch(output_box)
    ax.text(4.75, 1.35, 'OUTPUT', ha='center', va='center',
            fontweight='bold', fontsize=10, color='white')
    ax.text(4.75, 1.05, 'SOFTMAX\nDefective / Clean',
            ha='center', va='center', fontsize=8, color='white')

    # ==================== ARROWS ====================
    # Input → Conv1
    ax.annotate('', xy=(3, 6.2), xytext=(3.5, 5.9),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='blue'))

    # Conv1 → Conv2
    ax.annotate('', xy=(4.75, 5.5), xytext=(4.75, 4.7),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='purple'))

    # Conv2 → Conv3
    ax.annotate('', xy=(4.75, 4.5), xytext=(4.75, 3.7),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='green'))

    # Conv3 → Flatten
    ax.annotate('', xy=(4.75, 3.5), xytext=(4.75, 2.7),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='blue'))

    # Flatten → Variants
    ax.annotate('', xy=(6, 2.5), xytext=(7, 4.0),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='orange', linestyle='-'))
    ax.annotate('', xy=(6, 2.5), xytext=(7, 6.0),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='orange', linestyle='-'))

    # Variants → Output
    ax.annotate('', xy=(8.5, 5.0), xytext=(6, 1.7),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='red', linestyle='--'))
    ax.annotate('', xy=(8.5, 3.0), xytext=(6, 1.7),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='red', linestyle='--'))

    # ==================== CONFIGURATION ====================
    config_text = (
        "MODEL CONFIGURATION:\n"
        "• Optimizer: Adam (lr=4.65e-4)\n"
        "• Loss: Binary Crossentropy\n"
        "• Batch Size: 8\n"
        "• Early Stopping: patience=450\n"
        "• Activation: ReLU (conv), Softmax (output)\n"
        "• Regularization: L2 (λ=0.01)"
    )

    ax.text(9, 1.5, config_text, fontsize=7,
            bbox=dict(boxstyle="round,pad=0.4", facecolor='#F5F5F5',
                     edgecolor='gray', alpha=0.8),
            ha='center', va='center')

    # ==================== LEGEND ====================
    legend_text = (
        "ARCHITECTURAL RATIONALE:\n"
        "• 3 conv blocks capture non-linear patterns\n"
        "• Progressive dropout (0.25→0.5) for regularization\n"
        "• Two variants test depth impact on performance\n"
        "• L2 regularization prevents overfitting"
    )

    ax.text(1, 1.5, legend_text, fontsize=7,
            bbox=dict(boxstyle="round,pad=0.4", facecolor='#E3F2FD',
                     edgecolor='#1976D2', alpha=0.8),
            ha='center', va='center')

    plt.tight_layout()
    return fig, ax

# Generation
def generate_cnn1d_diagram():
    print("Generating CNN1D architecture diagram...")

    fig, ax = create_cnn1d_architecture_diagram()
    plt.savefig('/content/cnn1d_architecture.png', dpi=300,
                bbox_inches='tight', facecolor='white', edgecolor='none',
                pad_inches=0.1)
    print("✅ CNN1D architecture diagram saved: /content/cnn1d_architecture.png")
    plt.show()

if __name__ == "__main__":
    generate_cnn1d_diagram()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch

def create_optimized_cnn1d_diagram():
    """Optimized CNN1D Architecture Diagram"""

    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.rcParams['font.size'] = 8

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.axis('off')

    # Title
    ax.text(5, 7.5, 'CNN1D ARCHITECTURE FOR DEFECT PREDICTION',
            ha='center', va='center', fontweight='bold', fontsize=11)

    # ==================== STANDARDIZATION STRATEGIES ====================
    ax.text(5, 6.8, 'STANDARDIZATION STRATEGIES', ha='center', va='center',
            fontweight='bold', fontsize=9, color='navy')

    # S1 Strategy
    s1_box = FancyBboxPatch((1, 6), 2.5, 0.7,
                           boxstyle="round,pad=0.02",
                           facecolor='#E3F2FD',
                           edgecolor='#1976D2', linewidth=1.5)
    ax.add_patch(s1_box)
    ax.text(2.25, 6.35, 'S1', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(2.25, 6.05, 'Global',
            ha='center', va='center', fontsize=8)

    # S2 Strategy
    s2_box = FancyBboxPatch((4, 6), 2.5, 0.7,
                           boxstyle="round,pad=0.02",
                           facecolor='#E8F5E8',
                           edgecolor='#388E3C', linewidth=1.5)
    ax.add_patch(s2_box)
    ax.text(5.25, 6.35, 'S2', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(5.25, 6.05, 'Per-version',
            ha='center', va='center', fontsize=8)

    # S3 Strategy
    s3_box = FancyBboxPatch((7, 6), 2.5, 0.7,
                           boxstyle="round,pad=0.02",
                           facecolor='#FFF3CD',
                           edgecolor='#FFA000', linewidth=1.5)
    ax.add_patch(s3_box)
    ax.text(8.25, 6.35, 'S3-ROBUST', ha='center', va='center',
            fontweight='bold', fontsize=9)
    ax.text(8.25, 6.05, 'Version + Global',
            ha='center', va='center', fontsize=8)

    # ==================== INPUT LAYER ====================
    input_box = FancyBboxPatch((1, 5), 2, 0.6,
                             boxstyle="round,pad=0.02",
                             facecolor='#F5F5F5',
                             edgecolor='gray', linewidth=1)
    ax.add_patch(input_box)
    ax.text(2, 5.3, 'INPUT', ha='center', va='center',
            fontweight='bold', fontsize=8)
    ax.text(2, 4.9, '20 CK metrics',
            ha='center', va='center', fontsize=7)

    # ==================== CONVOLUTIONAL BLOCKS ====================
    # Conv Block 1
    conv1_box = FancyBboxPatch((3.5, 4.5), 2, 0.6,
                              boxstyle="round,pad=0.02",
                              facecolor='#E1BEE7',
                              edgecolor='#7B1FA2', linewidth=1.5)
    ax.add_patch(conv1_box)
    ax.text(4.5, 4.8, 'CONV1D 1', ha='center', va='center',
            fontweight='bold', fontsize=8)
    ax.text(4.5, 4.5, '256 filters',
            ha='center', va='center', fontsize=7)

    # Conv Block 2
    conv2_box = FancyBboxPatch((3.5, 3.7), 2, 0.6,
                              boxstyle="round,pad=0.02",
                              facecolor='#C8E6C9',
                              edgecolor='#388E3C', linewidth=1.5)
    ax.add_patch(conv2_box)
    ax.text(4.5, 4.0, 'CONV1D 2', ha='center', va='center',
            fontweight='bold', fontsize=8)
    ax.text(4.5, 3.7, '256 filters',
            ha='center', va='center', fontsize=7)

    # Conv Block 3
    conv3_box = FancyBboxPatch((3.5, 2.9), 2, 0.6,
                              boxstyle="round,pad=0.02",
                              facecolor='#BBDEFB',
                              edgecolor='#1976D2', linewidth=1.5)
    ax.add_patch(conv3_box)
    ax.text(4.5, 3.2, 'CONV1D 3', ha='center', va='center',
            fontweight='bold', fontsize=8)
    ax.text(4.5, 2.9, '256 filters',
            ha='center', va='center', fontsize=7)

    # MaxPooling annotation
    ax.text(6, 3.7, 'MAXPOOLING\n& DROPOUT',
            ha='center', va='center', fontsize=7,
            bbox=dict(boxstyle="round,pad=0.3", facecolor='#FFF3CD'))

    # ==================== FLATTEN LAYER ====================
    flatten_box = FancyBboxPatch((3.5, 2.1), 2, 0.6,
                               boxstyle="round,pad=0.02",
                               facecolor='#FFE0B2',
                               edgecolor='#F57C00', linewidth=1.5)
    ax.add_patch(flatten_box)
    ax.text(4.5, 2.4, 'FLATTEN', ha='center', va='center',
            fontweight='bold', fontsize=8)

    # ==================== DENSE LAYER VARIANTS ====================
    # CNN1D-Simple Variant
    simple_box = FancyBboxPatch((7, 4.2), 2, 0.8,
                              boxstyle="round,pad=0.02",
                              facecolor='#E8F5E8',
                              edgecolor='#388E3C', linewidth=1.5)
    ax.add_patch(simple_box)
    ax.text(8, 4.6, 'CNN1D-SIMPLE', ha='center', va='center',
            fontweight='bold', fontsize=8)
    ax.text(8, 4.3, '1 Dense Layer\n256 neurons',
            ha='center', va='center', fontsize=7)

    # CNN1D-Dense Variant
    dense_box = FancyBboxPatch((7, 3.0), 2, 0.8,
                             boxstyle="round,pad=0.02",
                             facecolor='#FFEBEE',
                             edgecolor='#D32F2F', linewidth=1.5)
    ax.add_patch(dense_box)
    ax.text(8, 3.4, 'CNN1D-DENSE', ha='center', va='center',
            fontweight='bold', fontsize=8)
    ax.text(8, 3.1, '2 Dense Layers\n256 neurons',
            ha='center', va='center', fontsize=7)

    # ==================== OUTPUT LAYER ====================
    output_box = FancyBboxPatch((4.5, 1), 2, 0.6,
                              boxstyle="round,pad=0.02",
                              facecolor='#4CAF50',
                              edgecolor='#2E7D32', linewidth=2)
    ax.add_patch(output_box)
    ax.text(5.5, 1.3, 'OUTPUT', ha='center', va='center',
            fontweight='bold', fontsize=9, color='white')
    ax.text(5.5, 1.0, 'BUG / NO BUG',
            ha='center', va='center', fontsize=8, color='white')

    # ==================== OPTIMIZED ARROWS ====================
    # Input → Conv1
    ax.annotate('', xy=(3, 5), xytext=(3.5, 4.8),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#1976D2'))

    # Conv1 → Conv2
    ax.annotate('', xy=(4.5, 4.5), xytext=(4.5, 4.0),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#7B1FA2'))

    # Conv2 → Conv3
    ax.annotate('', xy=(4.5, 3.7), xytext=(4.5, 3.2),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#388E3C'))

    # Conv3 → Flatten
    ax.annotate('', xy=(4.5, 2.9), xytext=(4.5, 2.4),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#1976D2'))

    # Flatten → Variants
    ax.annotate('', xy=(5.5, 2.1), xytext=(7, 3.5),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#F57C00', linestyle='-'))
    ax.annotate('', xy=(5.5, 2.1), xytext=(7, 4.7),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#F57C00', linestyle='-'))

    # Variants → Output
    ax.annotate('', xy=(8, 3.8), xytext=(6.5, 1.6),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#D32F2F'))
    ax.annotate('', xy=(8, 2.6), xytext=(6.5, 1.6),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#D32F2F'))

    # ==================== CONFIGURATION ====================
    config_text = (
        "ARCHITECTURE:\n"
        "• 3 Conv1D blocks (256 filters)\n"
        "• Kernel size: 2\n"
        "• Dropout: 0.25→0.5\n"
        "• L2 regularization\n"
        "• Two variants compared"
    )

    ax.text(8, 1.5, config_text, fontsize=7,
            bbox=dict(boxstyle="round,pad=0.4", facecolor='#F5F5F5',
                     edgecolor='gray'),
            ha='center', va='center')

    plt.tight_layout()
    return fig, ax

# Generation
def generate_optimized_cnn1d_diagram():
    print("Generating optimized CNN1D diagram...")

    fig, ax = create_optimized_cnn1d_diagram()
    plt.savefig('/content/cnn1d_optimized.png', dpi=300,
                bbox_inches='tight', facecolor='white', edgecolor='none',
                pad_inches=0.1)
    print("✅ Optimized CNN1D diagram saved: /content/cnn1d_optimized.png")
    plt.show()

if __name__ == "__main__":
    generate_optimized_cnn1d_diagram()